# Sesión 2 · Programación científica: de un cálculo a una función

**Programación avanzada en Python aplicada al análisis social** · viernes 4 de diciembre de 2026

Objetivos:

1. Calcular con columnas enteras de números usando arrays de NumPy, sin bucles.
2. Convertir un cálculo en una **función** con nombre, documentación y comprobaciones.
3. Aplicar esas funciones a una encuesta real, teniendo en cuenta los pesos.

Ejemplo conductor: medir la **desigualdad de renta**. En Aprendizaje estadístico visteis *qué* estadísticos usar; aquí nos ocupamos de *cómo programarlos bien*.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

## 1. Arrays: calcular con columnas enteras

Una lista de Python guarda valores; un **array** de NumPy, además, sabe hacer cuentas con todos a la vez.

In [ ]:
import numpy as np

rentas = np.array([9_500, 14_200, 18_000, 22_500, 31_000, 64_000])

print(rentas / 12)             # la operación se aplica a cada elemento
print(rentas.mean())
print(np.median(rentas))

Con una lista habría hecho falta un bucle. Comparad:

In [ ]:
lista = [9_500, 14_200, 18_000, 22_500, 31_000, 64_000]

mensual = []
for r in lista:
    mensual.append(r / 12)

print(mensual)

Escribir la operación sobre el array completo se llama **vectorizar**. El código es más corto, se lee mejor y es mucho más rápido cuando hay muchos datos.

### Filtrar con condiciones

Una comparación sobre un array devuelve un array de `True` y `False`, que sirve para quedarse con una parte. Es la misma idea que usamos ayer para filtrar filas de una tabla.

In [ ]:
es_alta = rentas > 20_000
print(es_alta)
print(rentas[es_alta])
print(es_alta.sum(), "rentas altas de", len(rentas))

### Funciones útiles de NumPy

In [ ]:
print(np.sort(rentas)[::-1])                 # ordenar (de mayor a menor)
print(rentas.cumsum())                       # suma acumulada
print(np.percentile(rentas, [25, 50, 75]))   # percentiles
print(np.round(np.log(rentas), 2))           # logaritmo de cada valor

### Medias ponderadas

En una encuesta, cada respuesta representa a un número distinto de personas: su **peso**. `np.average` calcula la media teniéndolo en cuenta.

In [ ]:
valores = np.array([10_000, 20_000, 60_000])
pesos = np.array([500, 400, 100])            # a cuántos hogares representa cada uno

print(valores.mean())                        # media simple
print(np.average(valores, weights=pesos))    # media ponderada

## 2. Un cálculo con nombre: funciones

Vamos a medir la desigualdad con la **ratio S80/S20**: la renta que suma el 20 % más rico dividida por la que suma el 20 % más pobre. Primero lo calculamos paso a paso.

In [ ]:
rng = np.random.default_rng(2026)                        # generador de números al azar, con semilla
rentas = rng.lognormal(mean=np.log(18_000), sigma=0.6, size=1_000)

ordenadas = np.sort(rentas)
n_grupo = len(ordenadas) // 5                            # tamaño de una quinta parte

mas_pobres = ordenadas[:n_grupo]
mas_ricos = ordenadas[-n_grupo:]

print(mas_ricos.sum() / mas_pobres.sum())

Funciona, pero si queremos repetirlo con otros datos habría que copiar y pegar esas líneas. Mejor darle **nombre**:

In [ ]:
def s80_s20(rentas):
    """Ratio entre la renta total del 20 % más rico y la del 20 % más pobre."""
    ordenadas = np.sort(rentas)
    n_grupo = len(ordenadas) // 5
    return ordenadas[-n_grupo:].sum() / ordenadas[:n_grupo].sum()


print(s80_s20(rentas))

Una buena función tiene:

- un **nombre** que dice qué calcula;
- una **explicación** entre comillas triples (el *docstring*), que es lo que muestra `help()`;
- **parámetros**: los datos que necesita;
- un **`return`** con el resultado.

In [ ]:
help(s80_s20)

### Comprobar que una función hace lo que debe

¿Cómo sabemos que está bien? Probándola con casos cuya respuesta conocemos. `assert` comprueba una condición y da error si no se cumple.

In [ ]:
todos_iguales = np.full(100, 20_000)                 # cien rentas idénticas
assert s80_s20(todos_iguales) == 1                   # sin desigualdad, la ratio vale 1

el_doble = np.r_[np.full(50, 10_000), np.full(50, 20_000)]
assert s80_s20(el_doble) == 2

print("Las comprobaciones pasan")

### Otra función: el índice de Gini

El índice de Gini resume la desigualdad en un número entre 0 (todos tienen lo mismo) y 1 (una sola persona lo tiene todo). Con las rentas ordenadas de menor a mayor:

$$G = \frac{2\sum_i i\,x_i}{n\sum_i x_i} - \frac{n+1}{n}$$

In [ ]:
def gini(rentas):
    """Índice de Gini de un conjunto de rentas no negativas (0 = igualdad total)."""
    x = np.sort(rentas)
    n = len(x)
    posicion = np.arange(1, n + 1)                   # 1, 2, 3, ..., n
    return 2 * np.sum(posicion * x) / (n * x.sum()) - (n + 1) / n


assert abs(gini(todos_iguales)) < 1e-9
print(round(gini(rentas), 3))

### Parámetros con valor por defecto

Un parámetro puede tener un valor por defecto, que se usa si no se indica otro. Así una misma función sirve para varios casos.

In [ ]:
def ratio_extremos(rentas, proporcion=0.2):
    """Renta del grupo más rico dividida por la del más pobre.

    'proporcion' es el tamaño de cada grupo: 0.2 da la S80/S20 y 0.1 la S90/S10.
    """
    if not 0 < proporcion <= 0.5:
        raise ValueError("La proporción debe estar entre 0 y 0.5")
    ordenadas = np.sort(rentas)
    n_grupo = int(len(ordenadas) * proporcion)
    return ordenadas[-n_grupo:].sum() / ordenadas[:n_grupo].sum()


print(round(ratio_extremos(rentas), 2))                    # S80/S20
print(round(ratio_extremos(rentas, proporcion=0.1), 2))    # S90/S10

La línea con `raise` detiene la función con un mensaje claro si alguien le pasa un valor sin sentido. Es preferible un error comprensible a un resultado equivocado.

## 3. Con datos de encuesta

La Encuesta de Condiciones de Vida (INE) es la fuente oficial sobre renta de los hogares. Usamos sus variables:

| Variable | Significado |
| --- | --- |
| `DB040` | Región |
| `DB090` | Peso del hogar: a cuántos hogares representa |
| `HY020` | Renta disponible del hogar en un año (euros) |
| `HX040` | Miembros del hogar |
| `HX240` | Unidades de consumo |

In [ ]:
import pandas as pd

ecv = pd.read_csv("datos/ecv_hogares.csv")
ecv.head(3)

Las columnas de una tabla de pandas son arrays de NumPy por dentro, así que nuestras funciones sirven tal cual.

In [ ]:
renta_equivalente = ecv["HY020"] / ecv["HX240"]          # renta por unidad de consumo

print("Hogares con renta negativa:", (renta_equivalente <= 0).sum())

validas = renta_equivalente[renta_equivalente > 0]
print("Gini:", round(gini(validas), 3))
print("S80/S20:", round(s80_s20(validas), 2))

Hemos tomado dos decisiones que conviene dejar escritas: dividir por las unidades de consumo y excluir las rentas negativas. Un comentario en el código, o mejor aún el *docstring* de una función, es el lugar para hacerlo.

### Aleatoriedad reproducible

`np.random.default_rng(semilla)` crea un generador de números al azar. Con la misma semilla, el «azar» se repite: quien ejecute vuestro código obtendrá exactamente lo mismo.

In [ ]:
rng = np.random.default_rng(123)
muestra = rng.choice(validas, size=500, replace=False)       # 500 hogares al azar

print(round(gini(muestra), 3), "en la muestra frente a", round(gini(validas), 3))

## Resumen

- Operad con **arrays completos**: `rentas / 12`, `rentas[rentas > 20_000]`, `np.average(x, weights=w)`.
- Cuando un cálculo funciona, **dadle nombre**: función, *docstring*, parámetros y `return`.
- **Comprobad** cada función con casos de respuesta conocida (`assert`).
- Fijad la **semilla** siempre que intervenga el azar.

`scipy.stats` ofrece en Python los contrastes e intervalos que conocéis de Aprendizaje estadístico (`ttest_ind`, `chi2_contingency`, `bootstrap`…); no los repetimos aquí.